<a href="https://colab.research.google.com/github/Ambaright/ST-554-Homework-8/blob/main/Baright_ST554_Homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Amanda Baright

Purpose: ST 554 Homework 7 and 8

Date: 03.31.2026

# Homework 7: Fitting MLR, Logistic Regression, Regression and Classification Trees, and Random Forests.

This homework will serve as practice for fitting MLR, logistic regression models (including penalized and regularized models), Regression and Classification Trees, and Random Forests. We will use the data from the UCI Machine Learning Repository that investigates wine quality. More info on the data can be found [here](https://archive.ics.uci.edu/dataset/186/wine+quality).

We have a few different input variables to work with and the common output variable is `quality`. However, for the purpose of this homework, we will treat `alcohol` as our target variable when fitting an MLR model and when fitting a logistic regression model we will use the type of wine as the response variable.

## Loading in the Data

First we want to read in the data. The data is given in two csv files (`winequality-red.csv` and `winequality-white.csv`) which we willl want to combine into one dataset. We will then create a new variable that represents the type of wine (red or white).

Note: The two csv files were downloaded from the UCI Machine Learning Repository and then uploaded into the Google Colab environment.

In [ ]:
# Packages for the assignment
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, RidgeCV, Ridge, ElasticNetCV, ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Now read in the data
red_wine = pd.read_csv('winequality-red.csv', sep=';')
white_wine = pd.read_csv('winequality-white.csv', sep=';')

print("Red wine head: ")
print(red_wine.head())
print("Red wine shape: ", red_wine.shape)

print("\nWhite wine head: ")
print(white_wine.head())
print("White wine shape: ", white_wine.shape)

Red wine head: 
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2   

Now we wish to combine these two datasets into one wine dataset. Before we do that, we will add in a new variable called `type` into each dataset to identify which type of wine we are working with. Here we will denote red wine as `type = 0` and white wine as `type = 1`, so the red wine will become our reference level.

We then combine these two datasets into one large wine dataset.

In [ ]:
red_wine['type'] = 0
white_wine['type'] = 1

# Combine the datasets
wine_df = pd.concat([red_wine, white_wine], axis=0).reset_index(drop=True)
wine_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0


We might want to drop any rows with missing values as a form of data cleaning.

In [ ]:
wine_df.dropna(inplace=True)

## Split the Data

We now want to split the data into a training and test set, using stratified sampling to make sure there is a similar proportion of white and red wines in the training and test sets. To complete the stratified sampling we can use the option `stratify = wine_df['type']` in our `train_test_split` function.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
 wine_df.drop(columns = ["alcohol", "type"]),
    wine_df[["alcohol", "type"]],
 test_size = 0.20,
 stratify = wine_df["type"],
 random_state = 42)

We can use the `.value_counts()` method to see the proportion of wine types in the full data so that we can compare to make sure we correctly did stratification by type using the y data with `type`.  Then we can look at the new `y_train` and `y_test` counts.

In [ ]:
round(wine_df["type"].value_counts()/len(wine_df["type"]), 3)

,count
type,
1,0.754
0,0.246


In [ ]:
round(y_train["type"].value_counts()/len(y_train["type"]), 3)

,count
type,
1,0.754
0,0.246


In [ ]:
round(y_test["type"].value_counts()/len(y_test["type"]), 3)

,count
type,
1,0.754
0,0.246


## Regression Task (`alcohol` as Response)

Now we will work on a few regression tasks where we use `alcohol` as a response.

We now want to scale the data since we are using regularized methods. This can be done by subtracting the mean and dividing by the sd. This is done for all of the `X_train` observations with a lambda function. We will then use the training means and sds for the test set too.

In [ ]:
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0)

X_train = X_train.apply(lambda x: (x - np.mean(x))/np.std(x), axis = 0)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4110,-0.780839,-0.235499,1.194170,1.229111,0.054220,2.108751,1.749101,0.623768,-0.174702,0.465371,0.201651,0.571351
3062,0.833361,-0.903661,1.194170,-0.887164,-0.142398,-1.098649,0.374488,-0.953600,-0.986520,-0.480979,0.201651,0.571351
5236,1.140827,-0.235499,0.150449,-0.125305,-1.069314,-0.423407,0.198256,0.020657,-0.611835,-0.278190,1.341998,0.571351
497,-0.012173,0.007469,0.011287,-0.612048,0.953048,0.701997,-0.048470,0.637023,0.637117,1.749702,-0.938696,-1.750237
826,0.218427,-0.417725,0.150449,-0.654374,-0.170487,-1.492540,-1.898910,0.139954,1.136697,0.735756,1.341998,-1.750237


We then want to apply the training means and stds when standardizing the test set.

In [ ]:
# quick function to standardize based on supplied means and stds
def my_std_fun(x, means, stds):
    return((x-means)/stds)
#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])
X_test.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
1739,-0.703973,-0.174757,0.150449,-0.675536,-0.310928,-0.592217,-0.682907,-0.655358,0.512221,-0.683768,-0.938696,0.571351
2845,0.602761,-0.842919,0.567938,-0.040654,-0.030045,0.589456,0.903185,0.206230,-0.237149,-0.886557,1.341998,0.571351
5282,-0.319639,-1.146628,0.637519,1.588878,-0.339017,2.755858,1.185157,0.908755,0.137536,1.682105,0.201651,0.571351
5822,-1.472639,0.554146,-1.449922,-0.908327,-0.760342,-1.380000,-1.141111,-1.298235,1.823620,-0.345786,-2.079043,0.571351
5237,-0.627106,-0.296241,0.011287,-0.633211,-1.181667,0.195565,-0.471428,-1.523573,0.137536,0.870949,1.341998,0.571351


### Train Models

Now that we did our standardization on the Training and Test sets we can finally start looking at some modeling with the training data.


#### Multiple Linear Regression Models

We first want to fit four different MLR models. Even though we don’t need to use CV on these models to determine a tuning parameter, we can use 5-fold CV on those to determine the best MLR model (using the training set alone). Here we use negative mean squared error as our scoring criteria. Each model will use `LinearRegression()` from `sklearn.linear_model` with the same y response of `alcohol`.

##### Model 1: MLR model with a few predictors

Our first model will be a basic MLR where we use a few of the predictors (`fixed acidity` and `pH`) and the `alcohol` response.

In [ ]:
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "pH"]],
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

##### Model 2: MLR with Six Predictors

The next model will look at six predictors `["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]`.

In [ ]:
cv_mlr2= cross_validate(
    LinearRegression(),
    X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

##### Model 3: Interaction Term

Now we want a model that includes some interaction terms. Here we might want to look at the interaction of `density` and `pH`.

In [ ]:
interaction = PolynomialFeatures(interaction_only=True, include_bias = False)
design_interaction = interaction.fit_transform(X_train[["density", "pH"]])
cv_mlr3 = cross_validate(
    LinearRegression(),
    design_interaction,
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

##### Model 4: Polynomial Terms

Now we want to include some polynomial terms. Here we might want to look at a quadratic and cubic polynomial for `pH` and `density`.

In [ ]:
poly = PolynomialFeatures(degree = (2,3), include_bias = False)
design_poly = poly.fit_transform(X_train[["pH", "density"]])
cv_mlr4 = cross_validate(
    LinearRegression(),
    design_poly,
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

##### Determine Best MLR
Now we need to select our best MLR model using the model metrics. The best model will have the smallest value. Here we see that `cv_mlr2` actually has the best performance out of the four MLR models, so the best MLR model is the multiple predictor mlr model.

In [ ]:
print(np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])),
      np.sqrt(-sum(cv_mlr4['test_score'])))


2.644459312290174 1.7105873527284496 1.9067439275237086 2.216542178945018


Here we can examine the coefficient estimates of the best MLR model.

In [ ]:
mlr_best = LinearRegression().fit\
        (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]], y_train["alcohol"])
print(mlr_best.intercept_)
print(mlr_best.coef_)

331.12071422845764
[-3.84306037e+00  1.51111909e+00  2.59752707e-01 -3.29729478e+02
  9.73457948e-01  2.90526813e-01]


#### LASSO Model

Now we want to fit a LASSO model with the same set of predictors as the best MLR (["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]), here we try by using all of the predictors as LASSO has some model selection built in.

In [ ]:
lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
         y_train["alcohol"])

We then will use CV to select the tuning parameter. Before we do that, we might want to look at all of the different alpha values and the CV errors.

In [ ]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0002, 0.5888],
       [0.0002, 0.5894],
       [0.0002, 0.5901],
       [0.0002, 0.5908],
       [0.0002, 0.5918],
       [0.0002, 0.5928],
       [0.0003, 0.5941],
       [0.0003, 0.5955],
       [0.0003, 0.5971],
       [0.0003, 0.599 ]])

Here we see that our best alpha value might be 0.002. We can double check this, as well as look at our coefficient estimates.

In [ ]:
print(lasso_mod.alpha_)
print(lasso_mod.intercept_)
print(lasso_mod.coef_)

0.0001695074764681405
306.1238896929869
[  -4.14972059    1.45421862    0.26089386 -304.1929072     0.93925822
    0.26478745]


Our cross validation does yeild a best tuning parameter of 0.00017. We can then fit our best LASSO model with this tuning parameter.

In [ ]:
lasso_best = Lasso(lasso_mod.alpha_).fit\
 (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],y_train["alcohol"])

#### Ridge Regression Model

We now want to find the best Ridge Regression model using CV, with a set of parameters of our choosing (again the same six as before). Here we also add a few extra optional values of alphas to fit across with the CV algorithm.

In [ ]:
ridge_mod = RidgeCV(cv=5, alphas = (0.05, 0.1, 0.2, 0.5, 1, 3, 10)) \
    .fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

We then might want to look at the best tuning parameter from the CV and the coefficient estimates. Here we see that the best tuning parameter is 0.05.

In [ ]:
print(ridge_mod.alpha_)
print(ridge_mod.intercept_)
print(ridge_mod.coef_)

0.05
138.658650138757
[  -7.4917454     1.14505534    0.34239795 -133.41884691    0.85520475
    0.09863325]


We then want to fit our best Ridge model, that has a tuning parameter of 0.05.

In [ ]:
ridge_best = Ridge(ridge_mod.alpha_).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

#### Elastic Net Model

Finally, we want to fit an Elastic Net model with a set of parameteres of our choosing, the same six. Again, Elastic Net models have a built in model selection component so we can just use all of our predictors and allow the algorithm to do the model selection. We then use CV to select our tuning parameters.

In [ ]:
elastic_mod = ElasticNetCV(cv=5,
                           random_state=0,
                           l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                           n_alphas = 50) \
    .fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

Now we want to check out what our best tuning parameters are and the coefficient estimates.

In [ ]:
print(elastic_mod.alpha_)
print(elastic_mod.l1_ratio_)
print(elastic_mod.coef_)

0.0001695074764681405
1.0
[  -4.14972059    1.45421862    0.26089386 -304.1929072     0.93925822
    0.26478745]


Here we see that our best alpha is 0.00017 and our best l1 ratio is 1. We will then use these values to fit our best Elastic Net model.

In [ ]:
elastic_best = ElasticNet(alpha = elastic_mod.alpha_, l1_ratio = elastic_mod.l1_ratio_).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

#### Regression Tree

We now want to fit a regression tree with the same set of six parameters and will use 5-fold CV to select the tuning parameters (`max_depth` and `min_samples_leaf`).

In [ ]:
# Set up the potential parameter values as a dictionary
parameters = {'max_depth': range(2,15),
              'min_samples_leaf': [3,5,10,50,100]}

# Use GridSearchCV to find the tuning parameters
tuning_reg_model = GridSearchCV(
    estimator = DecisionTreeRegressor(),
    param_grid = parameters,
    cv = 5,
    scoring = 'neg_mean_squared_error')

tuning_reg_model.fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

# Inspect the best tuning parameters
print(tuning_reg_model.best_score_, tuning_reg_model.best_params_)

-0.4665136566467895 {'max_depth': 8, 'min_samples_leaf': 10}


The `neg_mean_squared_error` scoring parameter is -0.466 and the best combo of tuning parameters is a `max_depth` of 8 and `min_samples_leaf` of 10. We then want to fit our best Regression Tree Model.

In [ ]:
reg_tree_best = DecisionTreeRegressor(max_depth = tuning_reg_model.best_params_['max_depth'],
                    min_samples_leaf = tuning_reg_model.best_params_['min_samples_leaf']).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

#### Random Forest Model

Next, we will fit a random forest model using the same six predictors. We will use 5-fold CV to determing the best `max_features`, `max_depth`, and `min_samples_leaf` tuning parameters.

In [ ]:
# Set up parameters for random forest
parameters_ref_rf = {"max_features": range(1,4), "max_depth": [3,4,5,10,15], "min_samples_leaf": [3,10,50,100]}
rf_reg_tune = GridSearchCV(estimator = RandomForestRegressor(),
                           param_grid = parameters_ref_rf,
                           cv = 5,
                           scoring = 'neg_mean_squared_error')
rf_reg_tune.fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

# Print for best estimators
print(rf_reg_tune.best_estimator_)

RandomForestRegressor(max_depth=15, max_features=3, min_samples_leaf=3)


The best Random Forest with Regression Trees has a `max_depth` of 15, `max_features` of 3, and `min_samples_leaf` of 3. We then fit this model.

In [ ]:
rf_reg_best = RandomForestRegressor(max_depth = rf_reg_tune.best_params_['max_depth'],
                                    max_features = rf_reg_tune.best_params_['max_features'],
                                    min_samples_leaf = rf_reg_tune.best_params_['min_samples_leaf']).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["alcohol"])

### Test Models

Now that we have the four selected models we want to compare their performance on the test set. We will do this by using RMSE as the model metric and then MAE as the model metric.

##### RMSE Model Metric

First up is testing the performance of our four best models on the test set using the RMSE. Here we will set up a dictionary of our best final models. We will then loop through each model and predict using our test set of predictors. We then calculate the RMSE with these predictions and print out the resulting dataframe.

In [ ]:
final_models = {
    "Best MLR": mlr_best,
    "LASSO": lasso_best,
    "Ridge": ridge_best,
    "Elastic Net": elastic_best,
    "Regression Tree": reg_tree_best,
    "Random Forest Regression Tree": rf_reg_best
}

test_results_rmse = []
for name, model in final_models.items():
    preds = model.predict(X_test[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]])
    rmse = np.sqrt(mean_squared_error(y_test["alcohol"], preds))
    test_results_rmse.append({"Model": name, "RMSE": rmse})

pd.DataFrame(test_results_rmse)

,Model,RMSE
0,Best MLR,0.764694
1,LASSO,0.768866
2,Ridge,0.920815
3,Elastic Net,0.768866
4,Regression Tree,0.690055
5,Random Forest Regression Tree,0.570766


To determine which model has the best performance on the test set, we want to select the model with the lowest RMSE. Here we see that they are all super close. However, the Random Forest Regression Tree model has the lowest RMSE and has the best performance on the test set.

##### MAE Model Metric

We will now repeat this with the MAE model metric. Again, the model with the best performance on the test set will have the lowest MAE.

In [ ]:
test_results_mae = []
for name, model in final_models.items():
    preds = model.predict(X_test[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]])
    mae = mean_absolute_error(y_test["alcohol"], preds)
    test_results_mae.append({"Model": name, "MAE": mae})

pd.DataFrame(test_results_mae)

,Model,MAE
0,Best MLR,0.593472
1,LASSO,0.596737
2,Ridge,0.753435
3,Elastic Net,0.596737
4,Regression Tree,0.502313
5,Random Forest Regression Tree,0.412177


Again, we see that the Random Forest Regression Tree model has the lowest MAE metric and has the best performance for the test set.

## Classification Task (Wine Type as Response)

We will now repeat the training and testing done previously but using logistic regression models with the wine `type` as the response. Again, we can recall that `type = 0` is for red wine and `type = 1` is for white wine. For these models, we will use the same set of predictors as in the Regression Task case and will standardize our training and test sets.

#### Train Models

Now we get into the section where we will train and fit the four different types of logistic regression models. Here when we train the models we will use the negative log-loss metric.

##### Multiple Logistic Regression

The first model type we want to fit is a Multiple Logistic Regression, where we will fit four different models in this family. We will then use 5-fold CV to determine the best model with `cross_validate` and will use `neg_log_loss` as the scoring criteria.

###### Model 1: MLR with a few predictors

The first model we will fit is one with a few parameters being used. Again, here we are using the `type` variable as the response. We also printed out the corresponding coefficient estimates.

In [ ]:
# Some packages we may need for the classification tasks
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import log_loss, accuracy_score

cv_mlogr1 = cross_validate(
    LogisticRegression(penalty = None),
    X_train[["chlorides", "pH"]],
    y_train["type"],
    cv = 5,
    scoring = "neg_log_loss")

###### Model 2: MLR with a Six parameters

The next model will look at six predictors `["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]`.

In [ ]:
cv_mlogr2= cross_validate(
    LogisticRegression(penalty = None),
    X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"],
    cv = 5,
    scoring = "neg_log_loss")

###### Model 3: Interaction

Now we want a model that includes some interaction terms. Here we will use the same interaction terms as the Regression case.


In [ ]:
cv_mlogr3 = cross_validate(
    LogisticRegression(),
    design_interaction,
    y_train["type"],
    cv = 5,
    scoring = "neg_log_loss")

###### Model 4: Polynomial Terms

Now we want to include some polynomial terms. Here we again use the same quadratic and cubic polynomial terms as the Regression case.

In [ ]:
cv_mlogr4 = cross_validate(
    LogisticRegression(),
    design_poly,
    y_train["type"],
    cv = 5,
    scoring = "neg_log_loss")

###### Determine Best Multiple Logistic Regression

Now we need to select our best MLR model using the model metric `neg_log_loss`. The best model will have the larger value. Here we see that the second model with six parameters actually has the best performance out of the four Multiple Logistic Regression models.

In [ ]:
[round(cv_mlogr1['test_score'].sum(),4),
 round(cv_mlogr2['test_score'].sum(),4),
 round(cv_mlogr3['test_score'].sum(),4),
 round(cv_mlogr4['test_score'].sum(),4)]

[np.float64(-1.6121),
 np.float64(-0.6805),
 np.float64(-2.4659),
 np.float64(-2.192)]

Now we can examine the coefficients for the best Multiple Logistic Regression Model (the full model).

In [ ]:
mlogr_best = LogisticRegression(penalty = None).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])
print("Intercept: ", mlogr_best.intercept_)
print("Coefficients: ", mlogr_best.coef_)

Intercept:  [31.16757286]
Coefficients:  [[-53.72882061 -11.2051976   11.72391714  28.83569293  -8.48333018
   -2.34606671]]


#### LASSO Model

We now move onto fitting a Logistic Regression LASSO model. Here it is important to note that `penalty = 'l1'` indicates a LASSO penalty and we need to specify the `solver = ` to be `liblinear` or `saga`. Again, we use the set of six predictors of our choosing, and since LASSO has some model selection built in we use all of the predictors.

In [ ]:
lasso_c_cv = LogisticRegressionCV(
    penalty='l1',
    solver='liblinear',
    Cs = 250,
    scoring = "neg_log_loss",
    cv=5,
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

We then might want to look at the best regularization strength C = 1/lambda) value and the coefficients.

In [ ]:
print("Regularization Strength: ", lasso_c_cv.C_)
print("Intercept: ", lasso_c_cv.intercept_)
print("Coefficients: ", lasso_c_cv.coef_)

Regularization Strength:  [17.25630432]
Intercept:  [30.50826053]
Coefficients:  [[-52.80615117 -10.97104602  11.62378084  28.40402498  -8.4171364
   -2.31281462]]


Now we can fit our best LASSO in the classification case.

In [ ]:
lasso_c_best = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C = lasso_c_cv.C_[0],
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

#### Ridge Regression Model

We now want to find the best Ridge Regression model using CV, with a set of parameters of our choosing. Here specifying `penalty = 'l2'` ensures we are doing Ridge Regression as the regularization parameter C is set to be equal to 1. This is the default penalty for the `sklearn.linear_model.LogisticRegression` package.

In [ ]:
ridge_c_cv = LogisticRegressionCV(
    penalty='l2',
    cv=5,
    Cs = 250,
    scoring = "neg_log_loss",
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

Again, we might want to look at the Regularization Strength and the coefficient estimates.

In [ ]:
print("Regularization Strength: ", ridge_c_cv.C_)
print("Intercept: ", ridge_c_cv.intercept_)
print("Coefficients: ", ridge_c_cv.coef_)

Regularization Strength:  [147.46015932]
Intercept:  [56.07188531]
Coefficients:  [[-48.82492597 -10.74913482  11.52265898   1.33309153  -8.27920361
   -2.26662791]]


Now we can fit our best Ridge Regression model in the classification task.

In [ ]:
ridge_c_best = LogisticRegression(
    penalty='l2',
    C = ridge_c_cv.C_[0],
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

#### Elastic Net Model

We now want to fit an Elastic Net Model with our choice of parameters. Again, as Elastic Net Models are a form of model selection we use all of the parameters. Here to specify Elastic Net we use `penalty = 'elasticnet'` and `solver = 'saga'`.

In [ ]:
elastic_c_cv = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
    cv=5,
    Cs = 250,
    scoring = "neg_log_loss",
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

Again, we might want to look at the Regularization Strength and the coefficient estimates.

In [89]:
print("Regularization Strength: ", elastic_c_cv.C_)
print("Intercept: ", elastic_c_cv.intercept_)
print("Coefficients: ", elastic_c_cv.coef_)

Regularization Strength:  [10000.]
Intercept:  [56.09572636]
Coefficients:  [[-48.24297008 -10.62187111  11.51880375   0.71616532  -8.26423895
   -2.24981117]]


Now we can fit our best Elastic Net model for the classification task.

In [91]:
elastic_c_best = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio = elastic_c_cv.l1_ratio_[0],
    C = elastic_c_cv.C_[0],
    random_state=42,
    max_iter=10000).fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

#### Classification Tree Model

Now we will fit a Classification Tree using `GridSearchCV` and 5-fold CV. We will use the `neg_log_loss` as the scoring metric and will find the optimial tuning parameters (`max_depth` and `min_samples_leaf`). We will include the standard six parameters we have been using.

In [93]:
clf_tree_tune = GridSearchCV(
    estimator = DecisionTreeClassifier(),
    param_grid = {'max_depth': range(2,15),
                  'min_samples_leaf': [3,5,10,50,100]},
    cv = 5,
    scoring = 'neg_log_loss').fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

# Inspect the best tuning parameters
print(clf_tree_tune.best_score_, clf_tree_tune.best_params_)

-0.15674450570838058 {'max_depth': 4, 'min_samples_leaf': 10}


The best Classification Tree has a `max_depth` of 4 and a `min_samples_leaf` of 10. We then fit our best Classification Tree model.

In [94]:
clf_tree_best = DecisionTreeClassifier(max_depth = clf_tree_tune.best_params_['max_depth'],
                    min_samples_leaf = clf_tree_tune.best_params_['min_samples_leaf']).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]],
    y_train["type"])

#### Random Forest Model with Classification Trees

Now we will do a Random Forest Model with Classification Trees, using 5-fold CV with `GridSearchCV` to determine the `max_features`, `max_depth`, and `min_samples_leaf`.

In [96]:
# Set up parameters for random forest
parameters_clf_rf = {"max_features": range(1,4), "max_depth": [3,4,5,10,15], "min_samples_leaf": [3,10,50,100]}
rf_clf_tune = GridSearchCV(estimator = RandomForestClassifier(),
                           param_grid = parameters_clf_rf,
                           cv = 5,
                           scoring = 'neg_log_loss')
rf_clf_tune.fit(X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["type"])

# Print for best estimators
print(rf_clf_tune.best_estimator_)

RandomForestClassifier(max_depth=10, max_features=3, min_samples_leaf=3)


The best Random Forest Model with Classification Trees has `max_depth` of 10, `max_features` of 3, and `min_samples_leaf` of 3 . We then fit this best model.

In [97]:
rf_clf_best = RandomForestClassifier(max_depth = rf_clf_tune.best_params_['max_depth'],
                                    max_features = rf_clf_tune.best_params_['max_features'],
                                    min_samples_leaf = rf_clf_tune.best_params_['min_samples_leaf']).fit\
    (X_train[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]]
         ,y_train["type"])

#### Test Models

Now that we have the four selected models we want to compare their performance on the test set. We will do this by using log-loss as the model metric and then accuracy as the model metric.

##### Log-loss Model Metric

First up is testing the performance of our four best models on the test set using the log-loss. Here we will set up a dictionary of our best final models. We will then loop through each model and predict using our test set of predictors. We then calculate the log-loss with these predictions and print out the resulting dataframe. For log-loss, a lower metric is an indicator of a better model.

In [100]:
class_models = {
    "Best Log-Reg": mlogr_best,
    "LASSO": lasso_c_best,
    "Ridge": ridge_c_best,
    "Elastic Net": elastic_c_best,
    "Classification Tree": clf_tree_best,
    "Random Forest Classification Tree": rf_clf_best
}

classification_results_ll = []

for name, model in class_models.items():
    y_prob = model.predict_proba(X_test[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]])

    loss = log_loss(y_test["type"], y_prob)

    classification_results_ll.append({
        "Model": name,
        "Log-Loss": round(loss, 4)
    })

# Display Results
pd.DataFrame(classification_results_ll)

,Model,Log-Loss
0,Best Log-Reg,0.1259
1,LASSO,0.1260
2,Ridge,0.1266
3,Elastic Net,0.1274
4,Classification Tree,0.1525
5,Random Forest Classification Tree,0.0562


Here we see that after evaluating the log-loss for each of the best models for the types, they're all super close. However, the Random Forest Classification Tree Model has the lowest log-loss metric.

##### Accuracy Metric

Now we will repeat the testing procedure, this time with the accuracy score metric. Here, the higher the accuracy the better the model.

In [101]:
classification_results_a = []

for name, model in class_models.items():
    y_pred = model.predict(X_test[["chlorides", "pH", "citric acid", "density", "sulphates", "fixed acidity"]])

    acc = accuracy_score(y_test["type"], y_pred)

    classification_results_a.append({
        "Model": name,
        "Accuracy": round(acc, 4)
    })

# Display Results
pd.DataFrame(classification_results_a)

,Model,Accuracy
0,Best Log-Reg,0.9585
1,LASSO,0.9585
2,Ridge,0.9592
3,Elastic Net,0.9577
4,Classification Tree,0.9608
5,Random Forest Classification Tree,0.9862


Here we see that our best model by the accuracy metric is also the Random Forest Classification Tree Model.